In [1]:
# =========================
# 0. Install dependencies
# =========================

!pip install -q transformers sentencepiece protobuf tiktoken scikit-learn pandas tqdm torch


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
from pathlib import Path

REPO_URL = input("Enter repository URL: ").strip()
REPO_DIR = Path("/content/AIR_repo")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already exists, skipping clone.")

%cd {REPO_DIR}

Enter repository URL: https://github.com/ayadssk/AIR_Group_Task.git
Cloning into '/content/AIR_repo'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 290 (delta 7), reused 0 (delta 0), pack-reused 254 (from 1)
Receiving objects: 100% (290/290), 43.73 MiB | 9.93 MiB/s, done.
Resolving deltas: 100% (154/154), done.
Updating files: 100% (73/73), done.
/content/AIR_repo


In [4]:
# =========================
# 1. Imports
# =========================
import os
import re
import json
import sys
import shutil
import subprocess
import time
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_cosine_schedule_with_warmup,
)

In [5]:
# =========================
# 2. Paths and config
# =========================

# Find repo root
REPO_ROOT = Path.cwd()

while not (
    (REPO_ROOT / "task2" / "reasoning_trace_build.py").exists()
    and (REPO_ROOT / "task2" / "scorer.py").exists()
):
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError(
            "Could not find repo root. Run this notebook from inside the cloned repository."
        )
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)

# Drive root for large data/results/checkpoints
BASE = "/content/drive/MyDrive/AIR_CheckThat"
DRIVE_ROOT = Path(BASE)

BASE_MODEL = "roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 4
INFERENCE_BATCH_SIZE = 1
EPOCHS = 3
LR = 1e-5
RANDOM_STATE = 42

MODEL_NAME = "roberta"
MODEL_TYPE = "baseline"
DISTILLATION_STRATEGY = "none"
PREPROCESSING_FAMILY = "provided_preprocessing"
EVIDENCE_INPUT = "no_evidence"

MODEL_ID = BASE_MODEL.replace("/", "_").replace("-", "_")
DISTILL_TAG = DISTILLATION_STRATEGY.replace(" ", "_").replace("-", "_")

# Scripts from Git repo
PREPROCESSOR_PATH = REPO_ROOT / "task2" / "reasoning_trace_build.py"
SCORER_PATH = REPO_ROOT / "task2" / "scorer.py"

# Main output dirs in Drive
RESULTS_ROOT = DRIVE_ROOT / "output" / "results"
PRED_ROOT = DRIVE_ROOT / "output" / "RM_prediction"
CKPT_ROOT = DRIVE_ROOT / "output" / "checkpoints"
TRAIN_JSONL_ROOT = DRIVE_ROOT / "output" / "training_data_for_RM"
RUNS_ROOT = DRIVE_ROOT / "runs"

for p in [RESULTS_ROOT, PRED_ROOT, CKPT_ROOT, TRAIN_JSONL_ROOT, RUNS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Scorer temporary files in Drive
SCORER_IO_DIR = PRED_ROOT
SCORER_INPUT = SCORER_IO_DIR / "clef_predictions.json"
SCORER_RESULT = SCORER_IO_DIR / "result.csv"
SCORER_IR = SCORER_IO_DIR / "per_sample_ir.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repo root:", REPO_ROOT)
print("Drive root:", DRIVE_ROOT)
print("Preprocessor exists:", PREPROCESSOR_PATH.exists())
print("Scorer exists:", SCORER_PATH.exists())
print("Device:", device)

Repo root: /content/AIR_repo
Drive root: /content/drive/MyDrive/AIR_CheckThat
Preprocessor exists: True
Scorer exists: True
Device: cuda


In [6]:
# =========================
# 3. Language registry
# =========================

LANGUAGES = [
    {
        "lang": "english",
        "train_path": DRIVE_ROOT / "data" / "english" / "english_train.json",
        "val_path": DRIVE_ROOT / "data" / "english" / "clef2026_gpt4_o_mini_val.json",
        "test_path": DRIVE_ROOT / "data" / "english" / "clef_2026_final_english_test.json",
        "train_jsonl": TRAIN_JSONL_ROOT / "english_train_provided_preprocessing.jsonl",
    },
    {
        "lang": "spanish",
        "train_path": DRIVE_ROOT / "data" / "spanish" / "spanish_train.json",
        "val_path": DRIVE_ROOT / "data" / "spanish" / "spanish_val.json",
        "test_path": DRIVE_ROOT / "data" / "spanish" / "clef_spanish_test_final.json",
        "train_jsonl": TRAIN_JSONL_ROOT / "spanish_train_provided_preprocessing.jsonl",
    },
    {
        "lang": "arabic",
        "train_path": DRIVE_ROOT / "data" / "arabic" / "clef2026_gpt4_o_mini_train_arabic.json",
        "val_path": DRIVE_ROOT / "data" / "arabic" / "clef2026_gpt4_o_mini_val_arabic.json",
        "test_path": DRIVE_ROOT / "data" / "arabic" / "clef_2026_final_arabic_test.json",
        "train_jsonl": TRAIN_JSONL_ROOT / "arabic_train_provided_preprocessing.jsonl",
    },
]

print("\nDATA CHECK")
print("=" * 100)
for lc in LANGUAGES:
    print(
        f"{lc['lang']:<8} | "
        f"train={lc['train_path'].exists()} | "
        f"val={lc['val_path'].exists()} | "
        f"test={lc['test_path'].exists()} | "
        f"jsonl={lc['train_jsonl']}"
    )


DATA CHECK
english  | train=True | val=True | test=True | jsonl=/content/drive/MyDrive/AIR_CheckThat/output/training_data_for_RM/english_train_provided_preprocessing.jsonl
spanish  | train=True | val=True | test=False | jsonl=/content/drive/MyDrive/AIR_CheckThat/output/training_data_for_RM/spanish_train_provided_preprocessing.jsonl
arabic   | train=True | val=True | test=True | jsonl=/content/drive/MyDrive/AIR_CheckThat/output/training_data_for_RM/arabic_train_provided_preprocessing.jsonl


In [7]:
# =========================
# 4. Helper functions
# =========================

def sync_if_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def get_hardware_name():
    if torch.cuda.is_available():
        return torch.cuda.get_device_name(0)
    return "CPU"


def get_parameter_counts(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    return all_params, trainable_params


def print_trainable_parameters(model):
    all_params, trainable_params = get_parameter_counts(model)
    print(
        f"trainable params: {trainable_params:,} || "
        f"all params: {all_params:,} || "
        f"trainable%: {100 * trainable_params / all_params:.2f}"
    )


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / (1024 ** 2)


def remove_label_pattern(text):
    text = re.sub(
        r"(\[?\s*Justification\s*\]?:?\s*)|(\[Label\]:\s*(True|False|Conflicting|Supports|Refutes|SUPPORTS|REFUTES|CONFLICTING))",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()

    text = text.replace("\n", " ")
    text = text.split("Label:")[0]
    return text.strip()


def build_input(claim, verdict, justification):
    """
    Provided preprocessing format:
    Claim + Verdict + Justification
    No evidence field.
    """
    return (
        f"Claim: {claim}\n"
        f"Verdict: {verdict}\n"
        f"Justification: {justification}"
    )


def parse_scorer_result(result_csv_path):
    with open(result_csv_path, "r", encoding="utf-8") as f:
        content = f.read()

    m_f1 = re.search(
        r"^macro avg,[0-9.]+,[0-9.]+,([0-9.]+)",
        content,
        re.MULTILINE,
    )
    m_r5 = re.search(
        r"^5,([0-9.]+)",
        content,
        re.MULTILINE,
    )

    macro_f1 = float(m_f1.group(1)) if m_f1 else np.nan
    recall_at5 = float(m_r5.group(1)) if m_r5 else np.nan

    return macro_f1, recall_at5

In [8]:
# =========================
# 5. Aggregation strategies
# =========================

def agg_top1(verdict_list, score_list):
    return verdict_list[int(np.argmax(score_list))]


def agg_majority_top3(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(3, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]


def agg_majority_top5(verdict_list, score_list):
    top_idx = np.argsort(score_list)[::-1][:min(5, len(verdict_list))]
    return Counter([verdict_list[i] for i in top_idx]).most_common(1)[0][0]


def agg_score_weighted(verdict_list, score_list):
    weights = torch.sigmoid(torch.tensor(score_list)).numpy()
    tally = {}

    for v, w in zip(verdict_list, weights):
        tally[v] = tally.get(v, 0.0) + float(w)

    return max(tally, key=tally.get)


AGGREGATIONS = [
    ("top1", agg_top1),
    ("majority_top3", agg_majority_top3),
    ("majority_top5", agg_majority_top5),
    ("score_weighted", agg_score_weighted),
]

AGG_FN_MAP = dict(AGGREGATIONS)

In [ ]:
# =========================
# 6. Dataset
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["model_input_text"].tolist()
        self.labels = dataframe["Class"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item

In [10]:
# =========================
# 7. RoBERTa verifier model
# =========================

class CustomClassifier(torch.nn.Module):
    def __init__(
        self,
        model_name,
        num_labels=1,
        hidden_dim=None,
        dropout_value=0.1,
        freeze_base_layer=False,
    ):
        super().__init__()

        self.model = AutoModel.from_pretrained(model_name)

        if freeze_base_layer:
            for param in self.model.parameters():
                param.requires_grad = False

        hidden_size = self.model.config.hidden_size

        if hidden_dim:
            self.classifier = torch.nn.Sequential(
                torch.nn.Linear(hidden_size, hidden_dim),
                torch.nn.ReLU(),
                torch.nn.Dropout(dropout_value),
                torch.nn.Linear(hidden_dim, num_labels),
            )
        else:
            self.classifier = torch.nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = pooled_output.float()

        logits = self.classifier(pooled_output)
        return logits

In [11]:
# =========================
# 8. Trainer
# =========================

class TrainerModule:
    def __init__(
        self,
        model,
        tokenizer,
        train_loader,
        val_loader,
        epochs,
        lr,
        output_dir,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.tokenizer = tokenizer

        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs = epochs

        self.optimizer = AdamW(self.model.parameters(), lr=lr, eps=1e-8)
        self.loss_fn = torch.nn.BCEWithLogitsLoss()

        total_steps = len(train_loader) * epochs
        warmup_steps = int(0.05 * total_steps)

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            warmup_steps,
            total_steps,
        )

        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def train(self):
        for epoch in range(self.epochs):
            print(f"\nEpoch {epoch + 1}/{self.epochs}")
            self.model.train()

            total_loss = 0
            total_acc = 0

            for batch in tqdm(self.train_loader):
                self.optimizer.zero_grad()

                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device).float()

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)

                assert not torch.isnan(logits).any(), "NaN in logits"

                loss = self.loss_fn(logits, labels)
                assert not torch.isnan(loss), "NaN loss"

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                self.optimizer.step()
                self.scheduler.step()

                total_loss += loss.item()

                preds = (torch.sigmoid(logits) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(labels.detach().cpu().numpy(), preds)

            print(f"Train Loss: {total_loss / len(self.train_loader):.4f}")
            print(f"Train Acc: {total_acc / len(self.train_loader):.4f}")

            self.evaluate(epoch)

    def evaluate(self, epoch):
        self.model.eval()

        total_loss = 0
        total_acc = 0

        with torch.no_grad():
            for batch in self.val_loader:
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["labels"].to(self.device).float()

                logits = self.model(input_ids, attention_mask).squeeze(-1)
                logits = torch.clamp(logits, -50, 50)

                loss = self.loss_fn(logits, labels)

                total_loss += loss.item()

                preds = (torch.sigmoid(logits) >= 0.5).detach().cpu().numpy()
                total_acc += accuracy_score(labels.detach().cpu().numpy(), preds)

        print(f"Val Loss: {total_loss / len(self.val_loader):.4f}")
        print(f"Val Acc: {total_acc / len(self.val_loader):.4f}")

        self.tokenizer.save_pretrained(self.output_dir)

        torch.save(
            self.model.state_dict(),
            self.output_dir / f"model_epoch_{epoch}.pt",
        )

In [12]:
# =========================
# 9. Prediction evaluator
# =========================

class VerifierEvaluator:
    def __init__(
        self,
        model_path,
        tokenizer_path,
        base_model,
        device="cuda",
    ):
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

        self.model = CustomClassifier(
            model_name=base_model,
            freeze_base_layer=False,
        )

        self.model.load_state_dict(
            torch.load(model_path, map_location=self.device)
        )

        self.model.to(self.device)
        self.model.eval()

    def score_text(self, text):
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        with torch.no_grad():
            logits = self.model(
                encoding["input_ids"].to(self.device),
                encoding["attention_mask"].to(self.device),
            )

        return float(logits[0][0].item())

In [13]:
# =========================
# 10. Score split helper
# =========================

def score_split(data, evaluator, split_name, lang):
    print(f"[{lang}] Scoring {split_name} split...")

    scored_samples = []
    total_trace_count = 0

    sync_if_cuda()
    start_time = time.perf_counter()

    for idx, sample in enumerate(tqdm(data, desc=f"{split_name}|{lang}")):
        claim = sample["claim"]

        verdict_list = []
        score_list = []
        justification_list = []

        traces = sample["Reasoning_traces"]
        total_trace_count += len(traces)

        for trace_idx, trace in enumerate(traces):
            justification = remove_label_pattern(trace)
            verdict = sample["Verdict_list"][trace_idx].lower()

            input_text = build_input(
                claim=claim,
                verdict=verdict,
                justification=justification,
            )

            score = evaluator.score_text(input_text)

            verdict_list.append(sample["Verdict_list"][trace_idx])
            justification_list.append(justification)
            score_list.append(score)

        scored_samples.append({
            "query_id": sample.get("query_id", idx),
            "Claim": claim,
            "Label": sample.get("label", ""),
            "verdict_list": verdict_list,
            "score_list": score_list,
            "justification_list": justification_list,
        })

    sync_if_cuda()
    end_time = time.perf_counter()

    inference_time_sec = end_time - start_time
    avg_time_claim_ms = (inference_time_sec / max(len(data), 1)) * 1000
    avg_time_reasoning_ms = (inference_time_sec / max(total_trace_count, 1)) * 1000
    avg_traces_claim = total_trace_count / max(len(data), 1)

    return {
        "scored_samples": scored_samples,
        "n_claims": len(data),
        "n_reasoning_traces": total_trace_count,
        "inference_time_sec": inference_time_sec,
        "avg_time_claim_ms": avg_time_claim_ms,
        "avg_time_reasoning_ms": avg_time_reasoning_ms,
        "avg_traces_claim": avg_traces_claim,
    }

In [14]:
# =========================
# 11. Save predictions helper
# =========================

def build_predictions(scored_samples, agg_fn):
    predictions = []

    for s in scored_samples:
        best_verdict = agg_fn(s["verdict_list"], s["score_list"])

        predictions.append({
            "query_id": s["query_id"],
            "Claim": s["Claim"],
            "Label": s.get("Label", ""),
            "Verdict_BoN": best_verdict,
            "BoN_Verdict_list": s["verdict_list"],
            "Reasoning_traces": s["justification_list"],
            "score_list": s["score_list"],
        })

    return predictions


def create_trec_file(predictions, trec_path, run_id):
    trec_path = Path(trec_path)
    trec_path.parent.mkdir(parents=True, exist_ok=True)

    with open(trec_path, "w", encoding="utf-8") as out:
        for sample in predictions:
            query_id = sample["query_id"]
            ranked = sorted(
                enumerate(sample["score_list"]),
                key=lambda x: x[1],
                reverse=True,
            )

            for rank, (trace_idx, score) in enumerate(ranked, start=1):
                out.write(
                    f"{query_id}\tQ0\t{query_id}_{trace_idx}\t{rank}\t{score:.6f}\t{run_id}\n"
                )

    return trec_path.exists()


def run_scorer(pred_path, result_path, ir_path):
    shutil.copy(pred_path, SCORER_INPUT)

    result = subprocess.run(
        [sys.executable, str(SCORER_PATH)],
        cwd=str(DRIVE_ROOT),
        capture_output=True,
        text=True,
    )

    print("Scorer return code:", result.returncode)
    print(result.stdout)
    print(result.stderr)

    result.check_returncode()

    shutil.copy(SCORER_RESULT, result_path)
    shutil.copy(SCORER_IR, ir_path)

    return parse_scorer_result(result_path)

In [15]:
# =========================
# 12. Main validation experiment loop
# =========================

all_results = []

hardware_name = get_hardware_name()
run_on_hardware = "GPU" if torch.cuda.is_available() else "CPU"

for lang_cfg in LANGUAGES:
    lang = lang_cfg["lang"]

    print("\n" + "=" * 100)
    print(f"LANGUAGE: {lang.upper()} | PROVIDED PREPROCESSING")
    print("=" * 100)

    # Run provided preprocessing if missing
    if lang_cfg["train_jsonl"].exists():
        print(f"[{lang}] JSONL exists, skipping preprocessing: {lang_cfg['train_jsonl']}")
    else:
        print(f"[{lang}] Running provided preprocessing...")
        subprocess.run(
            [
                sys.executable,
                str(PREPROCESSOR_PATH),
                "--input",
                str(lang_cfg["train_path"]),
                "--output",
                str(lang_cfg["train_jsonl"]),
            ],
            cwd=str(DRIVE_ROOT),
            check=True,
        )

    train_df = pd.read_json(lang_cfg["train_jsonl"], lines=True)

    # provided preprocessing usually creates input_text
    if "input_text" not in train_df.columns:
        raise KeyError(f"'input_text' missing in {lang_cfg['train_jsonl']}")

    train_df["model_input_text"] = train_df["input_text"]

    print(f"[{lang}] Training examples: {len(train_df):,}")
    print(f"[{lang}] Class distribution: {dict(train_df['Class'].value_counts())}")

    with open(lang_cfg["val_path"], "r", encoding="utf-8") as f:
        val_data = json.load(f)

    print(f"[{lang}] Validation samples: {len(val_data):,}")

    ckpt_id = f"{MODEL_ID}_{MODEL_TYPE}_{DISTILL_TAG}_{PREPROCESSING_FAMILY}_{EVIDENCE_INPUT}_{lang}"
    MODEL_DIR = CKPT_ROOT / lang / ckpt_id
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    expected_ckpts = [
        MODEL_DIR / f"model_epoch_{e}.pt" for e in range(EPOCHS)
    ]

    training_time_sec = np.nan
    training_status = "skipped_existing_checkpoint"

    if all(p.exists() for p in expected_ckpts):
        print(f"[{lang}] All checkpoints found, skipping training.")
    else:
        training_status = "trained"

        train_split, dev_split = train_test_split(
            train_df,
            test_size=0.2,
            stratify=train_df["Class"],
            random_state=RANDOM_STATE,
        )

        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

        train_dataset = TextDataset(train_split, tokenizer, MAX_LENGTH)
        dev_dataset = TextDataset(dev_split, tokenizer, MAX_LENGTH)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE)

        model = CustomClassifier(model_name=BASE_MODEL, freeze_base_layer=False)
        print_trainable_parameters(model)

        trainer = TrainerModule(
            model=model,
            tokenizer=tokenizer,
            train_loader=train_loader,
            val_loader=dev_loader,
            epochs=EPOCHS,
            lr=LR,
            output_dir=MODEL_DIR,
        )

        sync_if_cuda()
        train_start = time.perf_counter()
        trainer.train()
        sync_if_cuda()
        train_end = time.perf_counter()

        training_time_sec = train_end - train_start

    MODEL_PATH = MODEL_DIR / f"model_epoch_{EPOCHS - 1}.pt"

    evaluator = VerifierEvaluator(
        model_path=MODEL_PATH,
        tokenizer_path=BASE_MODEL,
        base_model=BASE_MODEL,
    )

    nr_params, trainable_params = get_parameter_counts(evaluator.model)
    model_size_mb = get_file_size_mb(MODEL_PATH)

    val_score_info = score_split(
        data=val_data,
        evaluator=evaluator,
        split_name="val",
        lang=lang,
    )

    for agg_name, agg_fn in AGGREGATIONS:
        run_id = f"{MODEL_ID}_{MODEL_TYPE}_{DISTILL_TAG}_{PREPROCESSING_FAMILY}_{EVIDENCE_INPUT}_{lang}_{agg_name}"

        RESULT_DIR = RESULTS_ROOT / lang / run_id
        PRED_DIR = PRED_ROOT / lang
        RUN_DIR = RUNS_ROOT / lang

        RESULT_DIR.mkdir(parents=True, exist_ok=True)
        PRED_DIR.mkdir(parents=True, exist_ok=True)
        RUN_DIR.mkdir(parents=True, exist_ok=True)

        PRED_PATH = PRED_DIR / f"val_predictions_{run_id}.json"
        RESULT_PATH = RESULT_DIR / f"val_result_{run_id}.csv"
        IR_PATH = RESULT_DIR / f"val_per_sample_ir_{run_id}.csv"
        TREC_PATH = RUN_DIR / f"val_trec_{run_id}.txt"

        print("\n" + "-" * 100)
        print(f"[{lang}] Aggregation: {agg_name}")
        print(f"Run ID: {run_id}")

        predictions = build_predictions(
            scored_samples=val_score_info["scored_samples"],
            agg_fn=agg_fn,
        )

        with open(PRED_PATH, "w", encoding="utf-8") as f:
            json.dump(predictions, f, indent=4, ensure_ascii=False)

        macro_f1, recall_at5 = run_scorer(
            pred_path=PRED_PATH,
            result_path=RESULT_PATH,
            ir_path=IR_PATH,
        )

        trec_created = create_trec_file(
            predictions=predictions,
            trec_path=TREC_PATH,
            run_id=run_id,
        )

        all_results.append({
            "Model": MODEL_NAME,
            "Type (baseline/experiment)": MODEL_TYPE,
            "Language": lang,
            "Base model": BASE_MODEL,
            "Student model": BASE_MODEL,
            "Teacher model": "none",
            "Distillation strategy": DISTILLATION_STRATEGY,
            "Preprocessing family": PREPROCESSING_FAMILY,
            "Evidence input": EVIDENCE_INPUT,
            "Aggregation": agg_name,
            "Run ID": run_id,

            "Val results.csv": str(RESULT_PATH),
            "Val per_sample_ir.csv": str(IR_PATH),
            "Val prediction JSON": str(PRED_PATH),

            "Nr. Of parameters": nr_params,
            "Trainable parameters": trainable_params,
            "Model size (MB)": round(model_size_mb, 2),

            "Training status": training_status,
            "Training time": round(training_time_sec, 2) if not np.isnan(training_time_sec) else "",

            "Val Avg time/claim": round(val_score_info["avg_time_claim_ms"], 2),
            "Val Avg. time/reasoning": round(val_score_info["avg_time_reasoning_ms"], 2),
            "Val Avg traces/claim": round(val_score_info["avg_traces_claim"], 2),

            "Val Macro F1": macro_f1,
            "Val Recall@5": recall_at5,

            "TREC file created": "yes" if trec_created else "no",
            "TREC file path": str(TREC_PATH) if trec_created else "",

            "Run on hardware (CPU/GPU)": run_on_hardware,
            "Hardware name": hardware_name,
            "Batch Size": BATCH_SIZE,
            "Inference batch size": INFERENCE_BATCH_SIZE,

            "Checkpoint path": str(MODEL_PATH),
            "n_val_claims": val_score_info["n_claims"],
            "n_val_reasoning_traces": val_score_info["n_reasoning_traces"],
        })

        print(f"Val Macro F1={macro_f1:.4f} | Val Recall@5={recall_at5:.4f}")
        print(f"Saved result.csv: {RESULT_PATH}")
        print(f"Saved per_sample_ir.csv: {IR_PATH}")


LANGUAGE: ENGLISH | PROVIDED PREPROCESSING
[english] Running provided preprocessing...
[english] Training examples: 31,433
[english] Class distribution: {0: np.int64(22775), 1: np.int64(8658)}
[english] Validation samples: 1,600


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

NameError: name 'TextDataset' is not defined

In [ ]:
# =========================
# 13. Save validation summary
# =========================

results_df = pd.DataFrame(all_results)

summary_path = DRIVE_ROOT / "output" / "roberta_provided_preprocessing_experiment_summary.csv"
results_df.to_csv(summary_path, index=False)

print("\n" + "=" * 120)
print("VALIDATION SUMMARY: ROBERTA + PROVIDED PREPROCESSING")
print("=" * 120)

display_cols = [
    "Model",
    "Language",
    "Preprocessing family",
    "Evidence input",
    "Aggregation",
    "Val Macro F1",
    "Val Recall@5",
    "Nr. Of parameters",
    "Model size (MB)",
    "Training time",
    "Val Avg time/claim",
    "Val Avg. time/reasoning",
    "Val Avg traces/claim",
    "TREC file created",
    "Run on hardware (CPU/GPU)",
    "Batch Size",
]

print(results_df[display_cols].to_string(index=False))

print("\nBEST CONFIG PER LANGUAGE BY VAL MACRO F1")
print("=" * 120)

for lang, grp in results_df.groupby("Language"):
    best = grp.loc[grp["Val Macro F1"].idxmax()]
    print(
        f"{lang:<10} | "
        f"aggregation={best['Aggregation']:<16} | "
        f"F1={best['Val Macro F1']:.4f} | "
        f"R@5={best['Val Recall@5']:.4f} | "
        f"checkpoint={best['Checkpoint path']}"
    )

print(f"\nSaved validation summary to: {summary_path}")

In [ ]:
# =========================
# 14. CodaBench helpers
# =========================

def save_codabench_file(predictions, lang, output_dir, task_prefix="Task2"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    lang_name_map = {
        "english": "English",
        "spanish": "Spanish",
        "arabic": "Arabic",
    }

    lang_name = lang_name_map[lang]

    json_name = f"{task_prefix}_Numerical_claims_{lang_name}.json"
    zip_name = f"{task_prefix}_Numerical_claims_{lang_name}.zip"

    json_path = output_dir / json_name
    zip_path = output_dir / zip_name

    clean_predictions = []

    for item in predictions:
        clean_predictions.append({
            "query_id": item["query_id"],
            "Claim": item["Claim"],
            "Label": item.get("Label", ""),
            "Verdict_BoN": item["Verdict_BoN"],
            "BoN_Verdict_list": item["BoN_Verdict_list"],
            "Reasoning_traces": item["Reasoning_traces"],
            "score_list": item["score_list"],
        })

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(clean_predictions, f, indent=4, ensure_ascii=False)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(json_path, arcname=json_name)

    print(f"Saved CodaBench JSON: {json_path}")
    print(f"Saved CodaBench ZIP : {zip_path}")

    return json_path, zip_path

In [ ]:
# =========================
# 15. Run best validation config on test
# =========================

summary_path = DRIVE_ROOT / "output" / "roberta_provided_preprocessing_experiment_summary.csv"
results_df = pd.read_csv(summary_path)

SUBMISSION_DIR = (
    DRIVE_ROOT
    / "codabench_submissions"
    / f"{MODEL_NAME}_{MODEL_TYPE}_{DISTILL_TAG}_{PREPROCESSING_FAMILY}_{EVIDENCE_INPUT}_best_val_config"
)

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

test_summary_rows = []

for lang_cfg in LANGUAGES:
    lang = lang_cfg["lang"]

    test_path = lang_cfg["test_path"]

    lang_results = results_df[results_df["Language"] == lang].copy()

    if len(lang_results) == 0:
        raise ValueError(f"No validation results found for language: {lang}")

    best = lang_results.loc[lang_results["Val Macro F1"].idxmax()]

    best_agg_name = best["Aggregation"]
    best_agg_fn = AGG_FN_MAP[best_agg_name]
    checkpoint_path = best["Checkpoint path"]

    print("\n" + "=" * 120)
    print(f"TEST RUN: ROBERTA + PROVIDED PREPROCESSING | {lang.upper()}")
    print("=" * 120)
    print(f"Model: {MODEL_NAME}")
    print(f"Base model: {BASE_MODEL}")
    print(f"Preprocessing: {PREPROCESSING_FAMILY}")
    print(f"Evidence input: {EVIDENCE_INPUT}")
    print(f"Selected aggregation: {best_agg_name}")
    print(f"Selected by Val Macro F1: {best['Val Macro F1']}")
    print(f"Selected Val Recall@5: {best['Val Recall@5']}")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Test path: {test_path}")

    if not Path(test_path).exists():
        raise FileNotFoundError(f"Test file not found: {test_path}")

    with open(test_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

    evaluator = VerifierEvaluator(
        model_path=checkpoint_path,
        tokenizer_path=BASE_MODEL,
        base_model=BASE_MODEL,
    )

    test_score_info = score_split(
        data=test_data,
        evaluator=evaluator,
        split_name="test",
        lang=lang,
    )

    test_predictions = build_predictions(
        scored_samples=test_score_info["scored_samples"],
        agg_fn=best_agg_fn,
    )

    json_path, zip_path = save_codabench_file(
        predictions=test_predictions,
        lang=lang,
        output_dir=SUBMISSION_DIR,
        task_prefix="Task2",  # Change to Task3 only if CodaBench explicitly requires Task3.
    )

    test_summary_rows.append({
        "Model": MODEL_NAME,
        "Model type": MODEL_TYPE,
        "Base model": BASE_MODEL,
        "Distillation strategy": DISTILLATION_STRATEGY,
        "Preprocessing family": PREPROCESSING_FAMILY,
        "Evidence input": EVIDENCE_INPUT,
        "Language": lang,
        "Dataset split": "test",
        "Run ID selected from validation": best["Run ID"],
        "Selected aggregation": best_agg_name,
        "Selected by Val Macro F1": best["Val Macro F1"],
        "Selected Val Recall@5": best["Val Recall@5"],
        "Checkpoint path": checkpoint_path,
        "Test path": str(test_path),
        "Test prediction JSON": str(json_path),
        "Test submission ZIP": str(zip_path),
        "Test samples": test_score_info["n_claims"],
        "Test reasoning traces": test_score_info["n_reasoning_traces"],
        "Test total time sec": round(test_score_info["inference_time_sec"], 2),
        "Test avg time/claim ms": round(test_score_info["avg_time_claim_ms"], 2),
        "Test avg time/reasoning ms": round(test_score_info["avg_time_reasoning_ms"], 2),
        "Test avg traces/claim": round(test_score_info["avg_traces_claim"], 2),
    })

test_summary_df = pd.DataFrame(test_summary_rows)

test_summary_path = SUBMISSION_DIR / "roberta_provided_preprocessing_test_submission_summary.csv"
test_summary_df.to_csv(test_summary_path, index=False)

print("\n" + "=" * 120)
print("TEST SUBMISSION SUMMARY")
print("=" * 120)
print(test_summary_df.to_string(index=False))
print("=" * 120)
print(f"Saved test summary to: {test_summary_path}")